In [2]:
from importlib import reload
from itertools import combinations

from typing import List

from semevalpolar.llm.prompt_utils import build_prompt
from semevalpolar.llm.data_utils import create_submission, parse_predictions, create_comparison_df, read_dataset
from semevalpolar.llm import custom_rules
from semevalpolar.llm.main import test_run, create_gen, create_response
from tqdm import tqdm
from semevalpolar.utils import get_project_root
from semevalpolar.ensemble.ensemble import run_base_model, proposal_veto_ensemble
import pandas as pd

import os
import ast

D:\projects\mldl\semevalpolar


In [3]:
batch_size = 10
data_path = os.path.join(get_project_root(), 'data', 'training', 'eng.csv')
gen = create_gen(data_path, batch_size=batch_size, randomize=True)
generator_list = list(gen)

In [3]:
combined_df = pd.concat(generator_list, ignore_index=True)

In [4]:
proposals = []
models = ["qwen/qwen3-max", "openai/gpt-5.2", "google/gemini-2.5-pro", "deepseek/deepseek-v3.2"]

In [ ]:
proposal = run_base_model(model=models[1], prompt=f"{get_project_root()}/src/semevalpolar/llm/prompt-ds-main-classifier.txt")

  0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
proposal1 = run_base_model(model=models[0], prompt="prompts/categories-2/contextdep.txt")
proposal2 = run_base_model(model=models[1], prompt="prompts/categories-2/lingeval.txt")
proposal3 = run_base_model(model=models[2], prompt="prompts/categories-2/veto.txt")
proposal4 = run_base_model(model=models[3], prompt="prompts/categories-2/stanceisolation.txt")

In [ ]:
proposals.append(proposal1)
proposals.append(proposal2)
proposals.append(proposal3)
proposals.append(proposal4)

In [ ]:
proposals

In [ ]:
proposals = [[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1],
            [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            [1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1]]

In [ ]:
predictions = proposal_veto_ensemble(proposals)

In [ ]:
comparison = combined_df[:20]["polarization"] == predictions

In [ ]:
comparison_df = pd.DataFrame({
	"text": combined_df[:20]["text"],
	"polarization": combined_df[:20]["polarization"],
	"prediction": predictions,
	"single-model": proposal,
})

In [ ]:
comparison_df

# Category 3

In [ ]:
proposal1 = run_base_model(model=models[1], prompt="prompts/categories-3/filteroutnews.txt")
proposal2 = run_base_model(model=models[1], prompt="prompts/categories-3/contextcompleteness.txt")
proposal3 = run_base_model(model=models[1], prompt="prompts/categories-3/opinionvspolarization.txt")
proposal4 = run_base_model(model=models[1], prompt="prompts/categories-3/rhetorical-intensity.txt")

In [ ]:
proposals = []
proposals.append(proposal1)
proposals.append(proposal2)
proposals.append(proposal3)
proposals.append(proposal4)

In [ ]:
proposals

In [ ]:
proposals_dict = {
	"base": proposal,
	"neutral": proposals[0],
	"context": proposals[1],
	"opinion": proposals[2],
	"rhetoric": proposals[3],
}

In [3]:
proposals_dict = {'base': [1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0],
 'neutral': [1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1],
 'context': [1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1],
 'opinion': [0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0],
 'rhetoric': [0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

In [26]:
def ensemble_decision(outputs: dict[str, list[int]]) -> list[int]:
    final = []

    for i in range(len(next(iter(outputs.values())))):
        if outputs["base"][i] == 0:
            final.append(0)
        elif outputs["neutral"][i] == 0:
            final.append(0)
        else:
            final.append(1)

    return final


In [27]:
final = ensemble_decision(proposals_dict)

In [28]:
pd.DataFrame({
	"Text": combined_df[:20]["text"],
	"GT": combined_df["polarization"][:20],
	"Preds": final,
	"Proposal": proposals_dict["base"]
})

,Text,GT,Preds,Proposal
0,Imagine blaming this on Israel instead of Hamas,1,1,1
1,Blinken is the Butcher of Blinken ButcherOfGaz...,1,1,1
2,"And as far as your gun control, Chicago knows ...",0,1,1
3,All the spoilers are on Fox News daily.,0,1,1
4,Are red states stealing from California?,1,1,1
5,ice cream ice cream ice cream,0,0,0
6,A Jewish state was created because no one else...,1,1,1
7,"""Sufficient evidence of voter fraud?"" Where? m...",1,1,1
8,Ceasefire is not in Putin vocabulary.,0,0,0
9,"Yeah, some IDF are indistinguishable from them.",1,1,1


In [9]:
final

[1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0]

In [15]:
proposal

NameError: name 'proposal' is not defined